### Imports and Setup

In [1]:
import os
import json
import gc
import time
import torch
import pandas as pd
from PIL import Image
from transformers import (
    Qwen2VLForConditionalGeneration,
    Qwen2VLProcessor,
    BitsAndBytesConfig,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from qwen_vl_utils import process_vision_info

os.environ["WANDB_DISABLED"] = "true"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

TUNING_VERSION = "A"
MODEL_NAME     = "qwen2vl_2b"
MODEL_ID       = "Qwen/Qwen2-VL-2B-Instruct"
CKPT_DIR       = f"../checkpoints/{MODEL_NAME}_tuning{TUNING_VERSION}"
LOG_DIR        = f"../logs/{MODEL_NAME}_tuning{TUNING_VERSION}"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)
print(f"✓ Checkpoint dir: {CKPT_DIR}")
print(f"✓ Log dir:        {LOG_DIR}")

/home/matty/miniconda3/envs/ML/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
VRAM:   6.1 GB
✓ Checkpoint dir: ../checkpoints/qwen2vl_2b_tuningA
✓ Log dir:        ../logs/qwen2vl_2b_tuningA


In [2]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### 01 - Hyperparameter Setup

In [3]:
HW_CONFIG = {
    "batch_size":             1,
    "gradient_accumulation":  8,
    "lora_r":                 4,       # reduced from 8
    "lora_alpha":             8,       # reduced from 16
    "lora_dropout":           0.05,
    "learning_rate":          2e-4,
    "warmup_ratio":           0.03,
    "epochs":                 3,
    "optim":                  "paged_adamw_8bit",
    "gradient_checkpointing": True,
    "bf16":                   True,
    "logging_steps":          10,
    "eval_steps":             100,
    "save_steps":             100,
    "save_total_limit":       2,
}

print("Hyperparameters:")
for k, v in HW_CONFIG.items():
    print(f"  {k}: {v}")

Hyperparameters:
  batch_size: 1
  gradient_accumulation: 8
  lora_r: 4
  lora_alpha: 8
  lora_dropout: 0.05
  learning_rate: 0.0002
  warmup_ratio: 0.03
  epochs: 3
  optim: paged_adamw_8bit
  gradient_checkpointing: True
  bf16: True
  logging_steps: 10
  eval_steps: 100
  save_steps: 100
  save_total_limit: 2


### 02 - Load Data

In [4]:
with open("../processed_data/splits/train_vA.json") as f:
    train_data = json.load(f)
with open("../processed_data/splits/val_vA.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")
print(f"\nSample prompt:\n{train_data[0]['user_text']}")
print(f"Answer: {train_data[0]['answer']}")

Train: 10918 samples
Val:   2300 samples

Sample prompt:
Question: [Label Question] What is C in the diagram?
A) fall  B) style branch  C) stem  D) standard
Answer: C) stem


### 03 - System prompt and format function

In [5]:
SYSTEM_PROMPT = (
    "You are a VLM specialized in scientific diagram understanding. "
    "Answer multiple-choice questions about diagrams by selecting the correct option. "
    "Respond ONLY with the letter and answer text in this exact format: X) answer text. "
    "Example: C) stem"
)

def format_sample(sample):
    # Images are NOT loaded here — loaded lazily in collate_fn
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image_path"]},  # just the path
                {"type": "text",  "text": sample["user_text"]},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}],
        },
    ]

train_dataset = [format_sample(s) for s in train_data]
val_dataset   = [format_sample(s) for s in val_data]
print(f"✓ Formatted {len(train_dataset)} train samples")
print(f"✓ Formatted {len(val_dataset)} val samples")

✓ Formatted 10918 train samples
✓ Formatted 2300 val samples


### 04 - Load Model

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_ID}...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
)
processor = Qwen2VLProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*28*28,
    max_pixels=512*28*28,   # ← limit max image size
)
processor.tokenizer.padding_side = "right"
print("✓ Model loaded")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

Loading Qwen/Qwen2-VL-2B-Instruct...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]/home/matty/miniconda3/envs/ML/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 729/729 [00:01<00:00, 585.99it/s]


✓ Model loaded
VRAM: 1.53 GB allocated


#### Patch the Model

In [7]:
from transformers.loss.loss_utils import ForCausalLMLoss
import torch.nn as nn

def patched_loss(logits, labels, vocab_size, num_items_in_batch=None, ignore_index=-100, **kwargs):
    # Keep in bf16 — do NOT upcast to float32
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous().to(logits.device)
    shift_logits = shift_logits.view(-1, vocab_size)
    shift_labels = shift_labels.view(-1)
    loss = nn.functional.cross_entropy(
        shift_logits, shift_labels, ignore_index=ignore_index, reduction="mean"
    )
    return loss

# Patch the model's loss function
model.loss_function = patched_loss
print("✓ Loss function patched to stay in bf16")

✓ Loss function patched to stay in bf16


In [9]:
print(type(model))
print(type(model.base_model))

<class 'transformers.models.qwen2_vl.modeling_qwen2_vl.Qwen2VLForConditionalGeneration'>
<class 'transformers.models.qwen2_vl.modeling_qwen2_vl.Qwen2VLModel'>


### 05 - LoRA Setup

In [8]:
peft_config = LoraConfig(
    r=HW_CONFIG["lora_r"],
    lora_alpha=HW_CONFIG["lora_alpha"],
    lora_dropout=HW_CONFIG["lora_dropout"],
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

print(f"Parameters before LoRA: {model.num_parameters():,}")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Parameters before LoRA: 2,208,985,600
trainable params: 1,089,536 || all params: 2,210,075,136 || trainable%: 0.0493


### 06 - Logging callback and collate function

In [9]:
class LoggingCallback(TrainerCallback):
    def __init__(self, log_path):
        self.log_path = log_path
        self.records  = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            record = {"step": state.global_step, **logs}
            self.records.append(record)
            pd.DataFrame(self.records).to_csv(self.log_path, index=False)

log_callback = LoggingCallback(f"{LOG_DIR}/training_log.csv")
print(f"✓ Logging to {LOG_DIR}/training_log.csv")

def collate_fn(examples):
    texts = [
        processor.apply_chat_template(ex, tokenize=False) for ex in examples
    ]
    # Load images here — only 1 image at a time per batch
    image_inputs = []
    for ex in examples:
        img_path = ex[1]["content"][0]["image"]  # the path string
        img = Image.open(img_path).convert("RGB")
        image_inputs.append(img)

    batch = processor(
        text=texts,
        images=image_inputs,
        return_tensors="pt",
        padding=True,
    )
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch

print("✓ Collate function ready")

✓ Logging to ../logs/qwen2vl_2b_tuningA/training_log.csv
✓ Collate function ready


### 07 - Training args and trainer

In [10]:
training_args = SFTConfig(
    output_dir=CKPT_DIR,
    num_train_epochs=HW_CONFIG["epochs"],
    per_device_train_batch_size=HW_CONFIG["batch_size"],
    per_device_eval_batch_size=HW_CONFIG["batch_size"],
    gradient_accumulation_steps=HW_CONFIG["gradient_accumulation"],
    gradient_checkpointing=HW_CONFIG["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=HW_CONFIG["learning_rate"],
    warmup_ratio=HW_CONFIG["warmup_ratio"],
    bf16=HW_CONFIG["bf16"],
    bf16_full_eval=True,        # ← added
    fp16_full_eval=False,       # ← added
    optim=HW_CONFIG["optim"],
    logging_steps=HW_CONFIG["logging_steps"],
    eval_steps=HW_CONFIG["eval_steps"],
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=HW_CONFIG["save_steps"],
    save_total_limit=HW_CONFIG["save_total_limit"],
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,
    max_grad_norm=1.0,
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    logging_dir=LOG_DIR,
    torch_empty_cache_steps=1,   # clear cache every step
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    processing_class=processor.tokenizer,
    callbacks=[log_callback],
)
print("✓ Trainer ready")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✓ Trainer ready


/tmp/ipykernel_82247/2234778608.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


### 08 - Baseline eval then train

In [15]:
# print("=== Baseline Evaluation (before fine-tuning) ===")
# baseline = trainer.evaluate()
# print(baseline)
# pd.DataFrame([baseline]).to_csv(f"{LOG_DIR}/baseline_metrics.csv", index=False)

print(f"\n=== Training {MODEL_NAME} — Tuning {TUNING_VERSION} ===")
trainer.train()

# Save final model
trainer.save_model(CKPT_DIR)
processor.save_pretrained(CKPT_DIR)
print(f"✓ Saved to {CKPT_DIR}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.



=== Training qwen2vl_2b — Tuning A ===


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 892.00 MiB. GPU 0 has a total capacity of 5.67 GiB of which 770.69 MiB is free. Including non-PyTorch memory, this process has 4.90 GiB memory in use. Of the allocated memory 3.69 GiB is allocated by PyTorch, and 1.09 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

### 09 - Plot loss curve

In [ ]:
import matplotlib.pyplot as plt

log_df = pd.read_csv(f"{LOG_DIR}/training_log.csv")

fig, ax = plt.subplots(figsize=(10, 4))
if "loss" in log_df.columns:
    ax.plot(log_df["step"], log_df["loss"], label="Train Loss")
if "eval_loss" in log_df.columns:
    eval_df = log_df.dropna(subset=["eval_loss"])
    ax.plot(eval_df["step"], eval_df["eval_loss"],
            label="Val Loss", linestyle="--")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title(f"{MODEL_NAME} Tuning {TUNING_VERSION} — Loss Curve")
ax.legend()
plt.tight_layout()
plt.savefig(f"{LOG_DIR}/loss_curve.png", dpi=150)
plt.show()

### 10 - Clear Memory

In [13]:
def clear_memory():
    for var in ["model", "trainer", "peft_config", "bnb_config", "processor"]:
        if var in globals():
            del globals()[var]
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"VRAM reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

clear_memory()

VRAM allocated: 2.77 GB
VRAM reserved:  4.30 GB
